In [2]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:

import argparse
import json
import math
import os
import random
import time
import urllib.request
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [4]:
SEED = 42   # fixed seed used for all random operations in this assignment


def set_all_seeds(seed: int) -> None:
    """Seed Python's random, NumPy, and PyTorch in one call."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ---------------------------------------------------------------------------
# Baseline configuration  (do not modify)
# ---------------------------------------------------------------------------

BASELINE_CONFIG = {
    "block_size":      128,   # context window length
    "embed_dim":        64,   # token embedding dimension D
    "num_heads":         4,   # number of attention heads
    "num_layers":        4,   # number of transformer blocks
    "mlp_dim":         128,   # MLP hidden dimension (2 × embed_dim)
    "dropout":         0.1,
    "lr":             3e-4,
    "batch_size":       64,
    "epochs":            5,
    "steps_per_epoch": 200,   # gradient steps per epoch
}

CHECKPOINT_EPOCHS = (1, 3, 5)
DATA_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master"
    "/data/tinyshakespeare/input.txt"
)



In [5]:
class CharDataset(Dataset):
    """
    Sliding-window character-level dataset.

    Each item is a pair (x, y) where:
      x = data[idx : idx + block_size]     input context
      y = data[idx+1 : idx + block_size+1] target (shifted by one position)
    """

    def __init__(self, data: torch.Tensor, block_size: int):
        self.data       = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx     : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


In [6]:

def get_dataset(
    data_root:  str   = "data",
    block_size: int   = 128,
    train_frac: float = 0.9,
):
    """
    Download TinyShakespeare if needed, build a character vocabulary, and
    return training / validation CharDataset objects with vocabulary metadata.

    Steps
    -----
    1.  Create data_root with os.makedirs if it does not exist.
    2.  Download DATA_URL to os.path.join(data_root, 'input.txt') if the
        file is not already present.  Use urllib.request.urlretrieve.
    3.  Read the full text from 'input.txt'.
    4.  Build the vocabulary: chars = sorted(set(text)), vocab_size = len(chars).
        Construct stoi (char→int) and itos (int→char) dicts.
    5.  Encode the entire text as a LongTensor using stoi.
    6.  Split: first train_frac of the data is training; rest is validation.
    7.  Wrap each split in CharDataset(split_data, block_size) and return.

    Args:
        data_root  (str):   Directory to store/read 'input.txt'.
        block_size (int):   Context window length.
        train_frac (float): Fraction of data for training.

    Returns:
        train_dataset (CharDataset)
        val_dataset   (CharDataset)
        vocab_size    (int)
        stoi          (dict[str, int])
        itos          (dict[int, str])
    """
    # TODO 1.1: implement
    os.makedirs(data_root, exist_ok = True)
    file_path = os.path.join(data_root, "input.txt")
    if os.path.isfile(file_path):
        pass
    else:
        urllib.request.urlretrieve(DATA_URL, file_path)

    text = ""

    with open(file_path, 'r') as file:
        text = file.read()

    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {}
    itos = {}
    for i,char in enumerate(chars):
        stoi[char] = i
        itos[i] = char
    
    encode = []
    for ch in text:
        encode.append(stoi[ch])

    data = torch.tensor(encode, dtype=torch.long)

    idx = int(len(data)*train_frac)
    train = data[:idx]
    val = data[idx:]

    train_dataset = CharDataset(train,block_size)
    val_dataset = CharDataset(val,block_size)

    return train_dataset, val_dataset, vocab_size, stoi, itos 



In [7]:
print(get_dataset())

(<__main__.CharDataset object at 0x119c2a230>, <__main__.CharDataset object at 0x119c2a020>, 65, {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}, {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 